In [3]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define and create the state folder
DRIVE_PATH = '/content/drive/MyDrive/advance_ml_project'
REGULARIZED_MODEL = DRIVE_PATH + '/tuned_with_regularization'
NON_REGULARIZED_MODEL = DRIVE_PATH + '/tuned_without_regularization'

assert(os.path.exists(DRIVE_PATH))
assert(os.path.exists(REGULARIZED_MODEL))
assert(os.path.exists(NON_REGULARIZED_MODEL))

BASE_MODEL = 'TechWolf/JobBERT-v2'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
"""
# =========================
# 0. Install
# =========================
!pip install -q sentence-transformers datasets scikit-learn

# =========================
# 1. Imports
# =========================
import torch
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 2. Load + preprocess CSV
# =========================
def clean_text(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = re.sub(r'[\r\n]+', ' ', x)
    x = re.sub(r'\s+', ' ', x)
    return x.strip()

df = pd.read_csv(DRIVE_PATH + "/dataset_5/jd_resume_score.csv")
df = df.map(clean_text).reset_index(drop=True)

candidate_cols = df.columns[[2, 4, 13, 14, 16]]
job_cols = df.columns[[28, 29, 30, 32, 33]]
score_col = df.columns[-1]

df["resume"] = df[candidate_cols].astype(str).agg(" ".join, axis=1)
df["jd"] = df[job_cols].astype(str).agg(" ".join, axis=1)
df["score"] = df[score_col].astype(float)

df = df[["jd", "resume", "score"]]

# =========================
# 3. Split
# =========================
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# =========================
# 4. Dataset
# =========================
def to_dataset(d):
    return Dataset.from_dict({
        "jd": d["jd"].tolist(),
        "resume": d["resume"].tolist(),
        "label": d["score"].tolist()
    })

train_dataset = to_dataset(train_df)
val_dataset = to_dataset(val_df)

# =========================
# 5. Eval fn
# =========================
def evaluate(model, d):
    emb1 = model.encode(d["jd"].tolist(), convert_to_tensor=True, device=device)
    emb2 = model.encode(d["resume"].tolist(), convert_to_tensor=True, device=device)
    preds = torch.nn.functional.cosine_similarity(emb1, emb2).cpu().numpy()
    true = d["score"].values
    return {
        "MSE": mean_squared_error(true, preds),
        "MAE": mean_absolute_error(true, preds)
    }

# =========================
# 6. Baseline
# =========================
base_model = SentenceTransformer(BASE_MODEL, device=device, trust_remote_code=True)
print("Baseline:", evaluate(base_model, val_df))
'''
# =========================
# 7. Train (WITH REG)
# =========================
model = SentenceTransformer(BASE_MODEL, device=device, trust_remote_code=True)

train_loss = losses.CosineSimilarityLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=REGULARIZED_MODEL,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    router_mapping={
        "jd": "query",
        "resume": "document"
    }
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
)

trainer.train()
trainer.save_model(REGULARIZED_MODEL)
model.save(REGULARIZED_MODEL)
'''

model_reg = SentenceTransformer(REGULARIZED_MODEL)
print("With REG:", evaluate(model_reg, test_df))

'''
# =========================
# 8. Train (NO REG)
# =========================
model_nr = SentenceTransformer(BASE_MODEL, device=device, trust_remote_code=True)
train_loss_nr = losses.CosineSimilarityLoss(model_nr)

args_nr = SentenceTransformerTrainingArguments(
    output_dir=NON_REGULARIZED_MODEL,
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    weight_decay=0.0,
    logging_steps=50,
    save_steps=500,
    eval_steps=500,
    router_mapping={
        "jd": "query",
        "resume": "document"
    }
)

trainer_nr = SentenceTransformerTrainer(
    model=model_nr,
    args=args_nr,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss_nr,
)

trainer_nr.train()
trainer_nr.save_model(NON_REGULARIZED_MODEL)
model_nr.save(NON_REGULARIZED_MODEL)
'''

model_nr = SentenceTransformer(NON_REGULARIZED_MODEL)
print("No REG:", evaluate(model_nr, test_df))
"""

<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2608/1888778698.py:29: SyntaxWarning: invalid escape sequence '\s'
  x = re.sub(r'\s+', ' ', x)


'\n# =========================\n# 0. Install\n# =========================\n!pip install -q sentence-transformers datasets scikit-learn\n\n# =========================\n# 1. Imports\n# =========================\nimport torch\nimport pandas as pd\nimport re\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.metrics import mean_squared_error, mean_absolute_error\nfrom sentence_transformers import SentenceTransformer, losses\nfrom sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments\nfrom datasets import Dataset\n\ndevice = "cuda" if torch.cuda.is_available() else "cpu"\n\n# =========================\n# 2. Load + preprocess CSV\n# =========================\ndef clean_text(x):\n    if pd.isna(x):\n        return x\n    x = str(x)\n    x = re.sub(r\'[\r\n]+\', \' \', x)\n    x = re.sub(r\'\\s+\', \' \', x)\n    return x.strip()\n\ndf = pd.read_csv(DRIVE_PATH + "/dataset_5/jd_resume_score.csv")\ndf = df.map(clean_text).reset_index(drop=

In [5]:
#test
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device, cos_sim

import torch

def encode_batch(jobbert_model, texts):
    features = jobbert_model.tokenize(texts)
    features = batch_to_device(features, jobbert_model.device)
    features["text_keys"] = ["anchor"]
    with torch.no_grad():
        out_features = jobbert_model.forward(features)
    return out_features["sentence_embedding"].cpu().numpy()

def encode(jobbert_model, texts, batch_size: int = 8):
    # Sort texts by length and keep track of original indices
    sorted_indices = np.argsort([len(text) for text in texts])
    sorted_texts = [texts[i] for i in sorted_indices]

    embeddings = []

    # Encode in batches
    for i in tqdm(range(0, len(sorted_texts), batch_size)):
        batch = sorted_texts[i:i+batch_size]
        embeddings.append(encode_batch(jobbert_model, batch))

    # Concatenate embeddings and reorder to original indices
    sorted_embeddings = np.concatenate(embeddings)
    original_order = np.argsort(sorted_indices)
    return sorted_embeddings[original_order]

def clean_text(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = re.sub(r'[\r\n]+', ' ', x)      # remove new lines / carriage returns
    x = re.sub(r'\s+', ' ', x)          # collapse multiple spaces
    x = x.strip()
    return x

job_list = ["""
Job Description: Senior Digital Marketing Manager

Company: NexaStream Tech
Location: Remote / Hybrid
Salary: $110k – $140k

Role Overview:
We are looking for a data-driven Senior Digital Marketing Manager to lead our growth initiatives. You will oversee a $2M annual ad spend across Google, Meta, and LinkedIn, focusing on B2B lead generation and CAC (Customer Acquisition Cost) optimization.

Key Responsibilities:

    Manage and optimize multi-channel performance marketing campaigns (SEM, Paid Social).

    Utilize HubSpot and Google Analytics 4 to track the customer journey.

    Lead a team of two specialists and coordinate with external creative agencies.

    Conduct A/B testing on landing pages to improve conversion rates by at least 15% YoY.

Requirements:

    6+ years of experience in B2B digital marketing.

    Proven track record of managing budgets over $1M.

    Expertise in SQL or advanced data visualization (Tableau/Looker) is a plus.
    """]

candidate_list = ["""
Name: Sarah Chen
Target Role: Senior Marketing Leadership

Summary
Strategic Growth Leader with 8 years of experience in B2B SaaS marketing. Expert at scaling paid media budgets and driving high-quality SQLs (Sales Qualified Leads) through data-backed experimentation.

Experience
Growth Marketing Manager | CloudScale Solutions (2021 – Present)

    Managed an annual performance marketing budget of $2.5M, achieving a 22% reduction in CAC over 18 months.

    Spearheaded cross-channel strategies on Google Ads and LinkedIn, resulting in a 40% increase in pipeline velocity.

    Daily power user of HubSpot and GA4; built custom attribution models using SQL and Tableau to visualize ROI.

    Mentored and led a team of 3 junior marketers and managed a $15k/month creative agency contract.

Digital Specialist | FinTech Global (2017 – 2021)

    Increased landing page conversion rates by 18% through rigorous A/B testing and Heatmap analysis.

Skills: SEM/PPC, B2B Strategy, HubSpot, SQL, Team Leadership, Budget Management ($1M+).
""", """
Name: Jason Miller
Target Role: Marketing Professional

Summary
Creative and energetic marketing professional with 3 years of experience in social media management and content creation. Passionate about brand storytelling and community engagement.

Experience
Social Media Coordinator | Spark Creative Agency (2023 – Present)

    Manage organic Instagram and TikTok accounts for 5 local retail clients.

    Designed "viral" aesthetic content using Canva and Adobe Express, increasing follower counts by 50%.

    Coordinated influencer outreach programs and managed a small monthly "boosted post" budget of $2,000.

    Wrote engaging blog posts and email newsletters for B2C audiences.

Marketing Intern | City Event Planners (2022 – 2023)

    Assisted in the setup of local community events and distributed promotional flyers.

    Monitored Facebook comments and responded to customer inquiries.

Skills: Content Writing, Social Media Management, Canva, Basic Meta Ads, Public Relations.
"""
]

device = "cuda" if torch.cuda.is_available() else "cpu"
assert(device == "cuda")

# Load the model
base_model = SentenceTransformer(BASE_MODEL, device=device)
non_reg_model = SentenceTransformer(NON_REGULARIZED_MODEL, device=device)
reg_model = SentenceTransformer(REGULARIZED_MODEL, device=device)

models = [base_model, non_reg_model, reg_model]

for model in models:
  # Get embeddings
  job_embeddings = encode(model, job_list)
  resume_embeddings = encode(model, candidate_list)

  # Calculate cosine similarity matrix
  similarities = cos_sim(job_embeddings, resume_embeddings)
  print(f"{similarities}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.9362, 0.7261]])


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

tensor([[0.9265, 0.7176]])
